# L28 · 均值（Mean）方差（Variance）标准化 — 描述性统计（Descriptive Statistics）、z-score 标准化与分布比较

**目标**：会算均值/方差/标准差（Standard Deviation，SD），并把数据标准化（z-score）。

**为什么对 Aurora 重要**：MFCC 特征喂进模型前要做 `zscore(x)` 标准化；BatchNorm 和 LayerNorm 本质上是带可学习参数 γ/β 的 z-score，跳过这步模型梯度（Gradient）容易爆炸或消失。

← **上一课**　[L27 · 概率基础](L27_probability_basics.ipynb)

> 上节课学习了 **概率基础**：事件、条件概率、独立性与大数定律。  
> 本课将探讨 **均值方差标准化**。

## 本课剧情：给一群数字画像

均值告诉你数据的重心在哪，标准差衡量数据偏离重心的典型距离。把两者合起来，z-score 可以把任意分布的数据拉到同一把尺上：均值 0、标准差 1。

## 1. 均值、方差、标准差

均值 μ = (1/N) Σxᵢ 是数据重心；方差 σ² = (1/N) Σ(xᵢ−μ)² 衡量散布程度；标准差 σ = √σ² 量纲与原数据相同，是最常用的散布度量。

**为什么神经网络需要标准化**：Aurora 的 mel 特征（取值约 0–80 dB）与 MFCC 系数（取值约 −20 到 +20）量纲完全不同。直接拼接送给线性层，梯度在高值维度上量级远大于低值维度，优化器步长失衡，收敛慢甚至发散。z-score 把每个维度独立映射到均值 0、标准差 1，使各维度梯度量级相当。BatchNorm 和 LayerNorm 是 z-score 的可学习推广：加入可训练的缩放 γ 和偏移 β，让网络在归一化（Normalization）之后自行决定最优值域——但前提依然是先做那步除以 σ 的归一化。

## 实验入口：从数组计算均值、方差、标准差

用 8 个数 `[2, 4, 4, 4, 5, 5, 7, 9]` 直接调用 `x.mean()`、`x.var()`、`x.std()`，观察 `var == std²` 的数值关系，以及均值如何等于所有值的算术平均。

In [ ]:
import numpy as np
x = np.array([2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 7.0, 9.0])
print('均值 mean :', x.mean())
print('方差 var  :', x.var())
print('标准差 std:', x.std())

## 动手观察：样本（Sample）量与样本标准差的稳定性

n=10 时估计的样本均值和 std 波动很大；n=10000 时趋向总体参数（μ=0, σ=1）。对比三个样本量，观察统计量的收敛过程。

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
for n in [10, 100, 10_000]:
    sample = rng.normal(0, 1, size=n)   # 从标准正态采样
    print(f'n={n:6d}  样本均值={np.mean(sample):+.3f}  样本std={np.std(sample):.3f}')


## 代码实验：重复 5 次，看样本 std 估计的抖动随 n 缩小

In [ ]:
import numpy as np

for n in [20, 200, 2000]:
    estimates = []
    for seed in range(5):
        rng = np.random.default_rng(seed)
        sample = rng.normal(0, 1, size=n)
        estimates.append(np.std(sample))
    print(f'n={n:4d} ->', np.round(estimates, 3),
          '均值std =', round(float(np.mean(estimates)), 3))


## 2. ✏️ 实现 z-score 标准化 `zscore(x)`

`(x − 均值) / 标准差`。标准化后均值≈0、标准差≈1。

**推理路线**：
1. 算均值 `mu = x.mean()`——减去均值后数据以 0 为中心，不再偏向任何一侧
2. 算标准差 `sigma = x.std()`——除以标准差后，偏差以"几倍 σ"为单位表达，不同量纲的特征可直接比较
3. 返回 `(x - mu) / sigma`——NumPy 广播对任意 shape 的 `x`（如 `(80, T)` 的 mel 矩阵）均有效

**参考输入输出**：`x=[2,4,4,4,5,5,7,9]` → 均值=5，std≈2，z-score≈[-1.5, -0.5, -0.5, -0.5, 0, 0, 1, 2]

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


### 写 `zscore` 前明确三件事

- 输入：一维数组 `x`，形状 `(N,)`，值域任意
- 关键步骤：`mean = x.mean()`，`std = x.std()`，`z = (x - mean) / std`
- 返回：与 `x` 等形状的数组，保证 `z.mean() ≈ 0`、`z.std() ≈ 1`

In [ ]:
def zscore(x):
    # ✏️ TODO: 返回标准化后的数组
    raise NotImplementedError("TODO: 返回 (x - μ) / σ 标准化数组")


In [ ]:
z = zscore(np.array([2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 7.0, 9.0]))
print('标准化后均值 ≈', round(float(z.mean()), 6), ' 标准差 ≈', round(float(z.std()), 6))
assert abs(z.mean()) < 1e-9 and abs(z.std() - 1.0) < 1e-9
print('✅ 通过：你会标准化特征了。')

## 3. 看一眼分布（直方图）

In [ ]:
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
plt.hist(rng.normal(0, 1, 5000), bins=40)
plt.title('histogram of 5000 samples'); plt.show()

**🔗 Aurora 连接**：z-score 标准化是 `src/aurora/` 特征流水线的**预处理步骤**，在特征提取器之外应用——`mfcc()` 本身只做 `mel_spectrogram → power_to_db → dct_ii`，不内置标准化。实际训练管线在调用 `mfcc()` 之后、送入模型之前，对批次特征执行 `z = (F - F.mean()) / F.std()` 消除各频段间绝对能量差异。PyTorch 的 `nn.BatchNorm1d` 在 mini-batch 维度动态估计均值/方差做归一化；`nn.LayerNorm` 在单个样本的特征维上做归一化——两者都是带可学习参数 γ/β 的 z-score 推广，跳过这步梯度容易在高方差特征维上爆炸。

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
F = rng.normal(0, 1, size=(10, 8))  # (n_mels, T) 类 mel 特征矩阵
F[2, :] *= 10                        # 第3频段能量异常偏大

# axis=0：每帧内做标准化（消除帧间差异）
z0 = (F - F.mean(axis=0, keepdims=True)) / (F.std(axis=0, keepdims=True) + 1e-8)
# axis=1：每频段内做标准化（消除频段基线偏移）
z1 = (F - F.mean(axis=1, keepdims=True)) / (F.std(axis=1, keepdims=True) + 1e-8)

print('axis=0 各帧   std（期望≈1）:', np.round(z0.std(axis=0), 2))  # axis=0 沿 mel 维压缩 → 每帧一个 std
print('axis=1 各频段 std（期望≈1）:', np.round(z1.std(axis=1), 2))  # axis=1 沿时间维压缩 → 每频段一个 std
print('→ 第3频段（高能量）经 axis=1 标准化后恢复正常。')


## 参数实验：两种轴方向的 z-score

对一批 mel 特征（形状 `(80, T)`，80 个频段、T 帧）分别做两种标准化：

- **沿时间轴 `axis=1`**：`(F - F.mean(axis=1, keepdims=True)) / F.std(axis=1, keepdims=True)`
  每个频段在自己的时间序列上独立标准化，消除各频段的绝对能量差异
- **沿特征轴 `axis=0`**：`(F - F.mean(axis=0, keepdims=True)) / F.std(axis=0, keepdims=True)`
  每一帧在 80 个频段间标准化，消除整帧的整体能量差异

用 `rng.normal(0, 1, (80, 100))` 生成模拟特征，验证：`axis=1` 后 `result.mean(axis=1)` 全近似 0；`axis=0` 后 `result.mean(axis=0)` 全近似 0。两种轴方向的选择直接影响网络看到的信息结构——前者强调频谱形状的时序变化，后者强调每帧内各频段的相对关系。

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
mel = rng.normal(0, 1, size=(4, 8))
mel[2, :] += 50   # 模拟 MFCC 第3频段基线偏移

mel_z = (mel - mel.mean(axis=1, keepdims=True)) / (mel.std(axis=1, keepdims=True) + 1e-8)

print('标准化前 行均值:', np.round(mel.mean(axis=1), 1))
print('标准化后 行均值:', np.round(mel_z.mean(axis=1), 1))   # 全部≈0
print('标准化后 行std :', np.round(mel_z.std(axis=1), 1))    # 全部≈1
print('→ 第3行（偏移+50）经 z-score 后与其他行幅度完全一致。')


## 本课收束

现在可以用 `zscore(x)` 把任意均值和方差的数组变换到均值 0、标准差 1。这个变换在 Aurora 的特征流水线里直接使用，MFCC 特征在送入分类器（Classifier）前会经过这一步。BatchNorm 和 LayerNorm 是 z-score 的可学习推广，参数 γ/β 让网络在标准化之后自行决定缩放和偏移。下一节（L29）将介绍常见概率分布：均匀分布、高斯分布、伯努利分布，以及它们的参数估计方式。

In [ ]:
# 小检查：手写 z-score，样本量越大越接近 μ=0、σ=1
import numpy as np

for n in [30, 300, 3000]:
    rng = np.random.default_rng(42)
    x = rng.normal(5.0, 2.0, size=n)   # 原始数据 μ=5, σ=2
    z = (x - x.mean()) / x.std()        # z-score
    print(f'n={n:4d} -> 标准化后 mean={z.mean():+.4f}, std={z.std():.4f}')
print('→ n 越大，z-score 后的均值越接近 0、std 越接近 1。')


---

→ **下一课**　[L29 · 常见概率分布](L29_distributions.ipynb)

> 下节课将学习 **常见概率分布**：均匀、高斯、伯努利：PDF / CDF 与采样。